<a href="https://colab.research.google.com/github/op1154/MBA-AI/blob/main/Gemma_4_rodando_Local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Executando o Gemma 4 Localmente

Para executar este código localmente, você precisará de um ambiente Python configurado em sua máquina. Se você pretende usar uma GPU, certifique-se de ter os drivers CUDA necessários e o PyTorch com suporte a CUDA instalados.

### 1. Crie um ambiente Python e instale as bibliotecas.

Primeiramente, recomenda-se criar um ambiente virtual para gerenciar suas dependências. Abra seu terminal ou prompt de comando e execute:

In [6]:
python -m venv venv_unsloth
python
#No Windows
venv_unsloth\Scripts\activate
pip install unsloth --upgrade
pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
pip install torch # Ensure this is compatible with your CUDA version if using GPU, e.g., `pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121` for CUDA 12.1

SyntaxError: invalid syntax (4166112858.py, line 1)

### 2. Prepare your script

Save the following code into a `.py` file (e.g., `gemma_inference.py`). You'll notice a few changes:

*   The `!pip install` lines are removed as you've already installed them.
*   The `from google.colab import userdata` and `GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')` lines are removed. If you need to use a Hugging Face token for gated models or higher rate limits, you would typically set it as an environment variable or pass it directly (e.g., `token = "YOUR_HF_TOKEN"`).
*   The `to("cuda")` part assumes you have a GPU available and properly configured with PyTorch. If you only have a CPU, you'll need to change this to `to("cpu")` or handle the device selection dynamically.

In [ ]:
import torch
from unsloth import FastModel
from transformers import TextStreamer

# You might need to set your Hugging Face token as an environment variable
# or pass it directly if you are using a gated model.
# For example: token = "hf_YOUR_TOKEN_HERE"

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, # None for auto detection
    max_seq_length = 1024,
    load_in_4bit = False,
    full_finetuning = False,
    # token = token, # Uncomment and set if needed
)

# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    # Dynamically set device to 'cuda' if available, otherwise 'cpu'
    device = "cuda" if torch.cuda.is_available() else "cpu"

    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True,
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to(device), # Use the determined device
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True)
    )

messages = [{
    "role" : "user",
    "content": [
        { "type": "text",  "text" : "O que é Inteligencia Artificial? "}
    ]
}]
do_gemma_4_inference(messages, max_new_tokens = 800)

### 3. Run the script

Finally, execute your Python script from the command prompt:

```bash
python gemma_inference.py
```

This will run the model and print the generated text to your console.